In [1]:
import pandas as pd

df = pd.read_csv("/content/phishing_site_urls.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/phishing_site_urls.csv'

In [ ]:
df.head()

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [ ]:
df['Label'].value_counts()

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.groupby('URL')['Label'].nunique().value_counts()
#checking the duplicate values are for same label or not

In [ ]:
# Remove URLs that have conflicting labels
conflicting_urls = df.groupby('URL')['Label'].nunique()
conflicting_urls = conflicting_urls[conflicting_urls > 1].index

df = df[~df['URL'].isin(conflicting_urls)]

# Remove exact duplicate rows
df = df.drop_duplicates()

# Reset row numbering
df = df.reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

In [ ]:
df['Label'].value_counts()
#check count of label(1.legitimate, 2.phishing)

In [ ]:
df['Label'].value_counts(normalize=True) * 100

In [ ]:
# Select 5,000 legitimate URLs
good_urls = df[df['Label'] == 'good'].sample(
    n=5000, random_state=42
)

# Select 5,000 phishing URLs
bad_urls = df[df['Label'] == 'bad'].sample(
    n=5000, random_state=42
)

# Combine both categories
df_sample = pd.concat([good_urls, bad_urls])

# Shuffle the dataset
df_sample = df_sample.sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("Sample dataset shape:", df_sample.shape)
print(df_sample['Label'].value_counts())

In [ ]:
print("Dataset shape:", df_sample.shape)

print("\nLabel distribution:")
print(df_sample['Label'].value_counts())

print("\nFirst 5 rows:")
display(df_sample.head())

In [ ]:
df_sample.to_csv(
    "phishguard_dataset.csv",
    index=False
)

In [ ]:
from google.colab import files
files.download("phishguard_dataset.csv")

In [ ]:
url = "https://google.com"

print("URL:", url)
print("Length:", len(url))
print("Number of dots:", url.count("."))
print("Number of hyphens:", url.count("-"))
print("Number of digits:", sum(c.isdigit() for c in url))

In [ ]:
def extract_features(url):
    return {
        'url_length': len(url),
        'dot_count': url.count('.'),
        'hyphen_count': url.count('-'),
        'https': int(url.startswith('https')),
        'digit_count': sum(c.isdigit() for c in url)
    }

In [ ]:
extract_features("https://google.com")

In [ ]:
features = df_sample['URL'].apply(extract_features)

features_df = pd.DataFrame(features.tolist())

display(features_df.head())

In [ ]:
df_final = features_df.copy()

df_final['Label'] = df_sample['Label'].values

display(df_final.head())

In [ ]:
df_final['Label'] = df_final['Label'].map({
    'good': 0,
    'bad': 1
})

display(df_final.head())

In [ ]:
print(df_final['Label'].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X = df_final.drop('Label', axis=1)
y = df_final['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print("Actual values:", y_test.values[:10])
print("Predicted values:", y_pred[:10])

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Phishing"]
))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_scaled = LogisticRegression(max_iter=1000)

model_scaled.fit(X_train_scaled, y_train)

In [ ]:
y_pred_scaled = model_scaled.predict(X_test_scaled)

print("Actual values:", y_test.values[:10])
print("New predictions:", y_pred_scaled[:10])


In [ ]:
from sklearn.metrics import accuracy_score

old_accuracy = accuracy_score(y_test, y_pred)
new_accuracy = accuracy_score(y_test, y_pred_scaled)

print("Original Model Accuracy:", old_accuracy * 100, "%")
print("Scaled Model Accuracy:", new_accuracy * 100, "%")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_scaled,
    target_names=["Legitimate", "Phishing"]
))

In [ ]:
import joblib

joblib.dump(model_scaled, 'phishguard_model.pkl')
joblib.dump(scaler, 'phishguard_scaler.pkl')

In [ ]:
test_url = "https://google.com"

new_features = extract_features(test_url)

new_df = pd.DataFrame([new_features])

new_df_scaled = scaler.transform(new_df)

prediction = model_scaled.predict(new_df_scaled)

if prediction[0] == 1:
    print("Potentially Phishing URL")
else:
    print("Likely Legitimate URL")

#false positive

In [ ]:
test_url = "http://example.com"

new_features = extract_features(test_url)
new_df = pd.DataFrame([new_features])
new_df_scaled = scaler.transform(new_df)

prediction = model_scaled.predict(new_df_scaled)

if prediction[0] == 1:
    print("Potentially Phishing URL")
else:
    print("Likely Legitimate URL")